# Failure Mode 8: Hallucinated Tool Call

> Before starting, read the [project README](../../README.md) for details on failure modes, scorers, and expectations.

The agent calls a tool that doesn't exist in its available tools list. The agent "hallucinates" a tool name — invoking something that was never defined in its tool set.

### `@scorer` for pure deterministic checks

The [Hallucinated Completion](../06_hallucinated_completion/06_hallucinated_completion.ipynb) notebook used `@scorer` to wrap an existing MLflow judge (`is_grounded()`) with custom context extraction. This notebook uses `@scorer` for pure deterministic logic — no LLM involved at all.

Use `@scorer` when the check is straightforward and doesn't require LLM reasoning. Tool existence is a set membership check (`called_tools ⊆ available_tools`) — no judgment needed, just a comparison.

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| Custom `@scorer` | Custom | No | Do all called tools exist in the available tools set? |

For a detailed explanation of this failure mode and how the scorer works, see [hallucinated_tool_call.md](hallucinated_tool_call.md).

### Prerequisites and setup

Complete the [project setup](../../README.md#setup) (dependencies, API keys, MLflow tracking) before running this notebook.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import Feedback, SpanType, Trace
from mlflow.genai.scorers import scorer
from tools import TRAVEL_AGENT_TOOLS
from utils import print_eval_results

load_dotenv()

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "agentic-evaluation")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'hallucinated_tool_call'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces for a travel booking agent. Three scenarios:

- **Hallucinated tool (fail):** Agent calls `check_passport` (doesn't exist in its tool set) before booking the flight. The agent should have skipped the passport check and proceeded directly with `search_and_book`.
- **Multiple hallucinated tools (fail):** Agent calls `book_transfer` and `arrange_pickup` — neither exists in its tool set. The agent should have completed only the flight booking and informed the user that transfers are outside its capabilities.
- **Valid tools only (pass):** Agent calls `search_and_book` — a tool that exists in its set.

In [ ]:
# --- Failing trace: agent calls a tool that doesn't exist ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def hallucinated_tool(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_tool_call", "expected_result": "fail"}
    )

    with mlflow.start_span(name="check_passport", span_type=SpanType.TOOL) as span:
        span.set_inputs({"user_id": "USR-12345"})
        span.set_outputs({"status": "valid", "expiry": "2030-01-01"})

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({"booking_id": "BK-456", "status": "confirmed"})

    return "Passport verified! Your flight is booked — NYC to London, July 20. Booking: BK-456."


# --- Failing trace: agent calls multiple tools that don't exist ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def multiple_hallucinated_tools(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_tool_call", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({"booking_id": "BK-456", "status": "confirmed"})

    with mlflow.start_span(name="book_transfer", span_type=SpanType.TOOL) as span:
        span.set_inputs({"airport": "LHR", "destination": "hotel"})
        span.set_outputs({"transfer_id": "TR-789", "status": "confirmed"})

    with mlflow.start_span(name="arrange_pickup", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "hotel", "time": "2026-07-20T14:00"})
        span.set_outputs({"pickup_id": "PU-321", "status": "confirmed"})

    return (
        "All set! Flight booked NYC to London (BK-456), "
        "airport transfer arranged (TR-789), and hotel pickup scheduled (PU-321)."
    )


# --- Passing trace: agent uses only valid tools ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def valid_tools_only(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_tool_call", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({
            "booking_id": "BK-456",
            "flight_id": "FL-123",
            "status": "confirmed",
        })

    return "Your flight is booked! NYC to London, July 20, FL-123. Booking: BK-456."


user_msg = [
    {"role": "user", "content": "Book me a flight from NYC to London on July 20."}
]
hallucinated_tool(user_msg)
multiple_hallucinated_tools([
    {
        "role": "user",
        "content": "Book me a flight from NYC to London and arrange airport transfer.",
    }
])
valid_tools_only(user_msg)
mlflow.flush_trace_async_logging()
print("Created 3 traces (2 fail, 1 pass)")

### Load traces

We fetch the Hallucinated Tool Call traces — two where the agent calls non-existent tools (one with a single hallucinated tool, one with multiple), and one where all tools are valid.

In [ ]:
halluc_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'hallucinated_tool_call'",
    return_type="list",
)

print(f"Traces found: {len(halluc_traces)}")
for t in halluc_traces:
    tags = t.info.tags or {}
    agent_spans = t.search_spans(span_type=SpanType.AGENT)
    available_tools = [
        tool["function"]["name"]
        for tool in (
            agent_spans[0].attributes.get("mlflow.chat.tools", [])
            if agent_spans
            else []
        )
    ]
    called_tools = [ts.name for ts in t.search_spans(span_type=SpanType.TOOL)]

    print(
        f"  [{tags.get('expected_result', '?')}] Input: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Output: {str(t.info.response_preview)[:80]}")
    print(f"    Available tools: {available_tools}")
    print(f"    Called tools: {called_tools}")
    print()

### Approach 1: Custom `@scorer` — tool existence check (deterministic)

The `@scorer` decorator lets you write any evaluation logic in Python. This scorer:
1. Reads the available tools from the agent span's `mlflow.chat.tools` attribute
2. Reads the called tools from the TOOL spans in the trace
3. Checks if every called tool name exists in the available set

No LLM needed — this is a pure set membership check.

In [ ]:
@scorer
def tool_existence_check(*, trace: Trace) -> Feedback:
    agent_spans = trace.search_spans(span_type=SpanType.AGENT)
    if not agent_spans:
        return Feedback(value="no", rationale="No agent span found in trace.")

    available_tools_raw = agent_spans[0].attributes.get("mlflow.chat.tools", [])
    available_names = {t["function"]["name"] for t in available_tools_raw}

    tool_spans = trace.search_spans(span_type=SpanType.TOOL)
    called_names = {ts.name for ts in tool_spans}

    hallucinated = called_names - available_names
    if hallucinated:
        return Feedback(
            value="no",
            rationale=(
                f"Hallucinated tool(s): {', '.join(sorted(hallucinated))}. "
                f"Available tools: {', '.join(sorted(available_names))}."
            ),
        )
    return Feedback(
        value="yes",
        rationale=f"All called tools ({', '.join(sorted(called_names))}) exist in the available set.",
    )

In [ ]:
with mlflow.start_run(run_name="hallucinated-tool-call-check") as run:
    results = mlflow.genai.evaluate(
        data=halluc_traces,
        scorers=[tool_existence_check],
    )

print_eval_results(results, "tool_existence_check", EXPERIMENT.experiment_id)

### Interpreting the results

- **Hallucinated tool** → `no` — `check_passport` doesn't exist in the agent's tool set. The rationale lists the hallucinated tool and the full available set.
- **Multiple hallucinated tools** → `no` — `book_transfer` and `arrange_pickup` don't exist. The scorer catches all hallucinated tools in one pass, not just the first one.
- **Valid tools only** → `yes` — `search_and_book` exists in the available set.

### `@scorer` vs `make_judge()`

MLflow provides two patterns for custom scorers:

- **`@scorer`** — write evaluation logic in Python. Use when the check is deterministic: set membership, threshold comparison, pattern matching. No LLM cost, instant, perfectly reproducible.
- **`make_judge()`** — delegate evaluation to an LLM. Use when the check requires reasoning or judgment that can't be reduced to a simple rule.

Tool existence is a set membership check — `@scorer` is the right choice. Compare this to [Graceful Refusal](../05_graceful_refusal/05_graceful_refusal.ipynb), where judging whether a refusal was appropriate requires understanding context and intent — that needs `make_judge()`.

**Cost tier:** This scorer is deterministic (Tier 1) — no LLM cost. Run it on all traces in every evaluation. See the [cost-effective evaluation strategy](../../README.md#cost-effective-evaluation-strategy) in the project README.

For full details on how the scorer works, see [hallucinated_tool_call.md](hallucinated_tool_call.md).